# 🚀 Phase 5 — Capstone Project
### PixelCraft Infotech | Web Scraping Course

## Project: Live Quote Dashboard

**What you'll build:**
- Scrape all 100 quotes from `quotes.toscrape.com` with full stealth setup
- Clean and store in SQLite database
- Export to Excel with formatting
- Schedule scraper to run automatically
- Build a Streamlit dashboard to browse quotes

**Skills used:** All previous phases combined!

---

## Part 1 — The Production Scraper

Combines everything: headless, UA rotation, random delays, WebDriverWait, error handling

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
import time
import random
import json
import csv
import pandas as pd
from datetime import datetime

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:121.0) Gecko/20100101 Firefox/121.0",
]

def create_driver():
    """Create a stealthy headless Chrome driver."""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(f"--user-agent={random.choice(USER_AGENTS)}")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    driver = webdriver.Chrome(options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

def human_delay(min_s=1.5, max_s=3.5):
    time.sleep(random.uniform(min_s, max_s))

def scrape_all_quotes():
    """Scrape all quotes from all pages with full production setup."""
    driver = create_driver()
    driver.get("http://quotes.toscrape.com/js")

    wait = WebDriverWait(driver, 15)
    all_quotes = []
    page = 1
    scrape_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    while True:
        print(f"Scraping page {page}...")
        try:
            quote_els = wait.until(
                EC.presence_of_all_elements_located((By.CLASS_NAME, "quote"))
            )
            for q in quote_els:
                try:
                    text   = q.find_element(By.CLASS_NAME, "text").text.strip()
                    author = q.find_element(By.CLASS_NAME, "author").text.strip()
                    tags   = [t.text for t in q.find_elements(By.CLASS_NAME, "tag")]

                    # Clean curly quotes
                    text = text.replace("\u201c", "").replace("\u201d", "")

                    all_quotes.append({
                        "quote": text,
                        "author": author,
                        "tags": ", ".join(tags),
                        "page": page,
                        "word_count": len(text.split()),
                        "scraped_at": scrape_time
                    })
                except StaleElementReferenceException:
                    print(f"  ⚠️ Stale element skipped")

            print(f"  ✅ {len(quote_els)} quotes")

            next_btn = wait.until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "li.next a"))
            )
            next_btn.click()
            page += 1
            human_delay(1.5, 3.0)

        except TimeoutException:
            print("  🛑 Done!")
            break

    driver.quit()
    print(f"\n✅ Total: {len(all_quotes)} quotes across {page} pages")
    return all_quotes


# Run the scraper
quotes_data = scrape_all_quotes()

---
## Part 2 — Store in SQLite Database

> SQLite is a lightweight database built into Python — no server needed  
> Perfect for storing scraped data persistently

In [ ]:
import sqlite3

def save_to_sqlite(data, db_name="quotes.db"):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    # Create table if it doesn't exist
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS quotes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            quote TEXT NOT NULL,
            author TEXT NOT NULL,
            tags TEXT,
            page INTEGER,
            word_count INTEGER,
            scraped_at TEXT
        )
    """)

    # Insert all rows
    cursor.executemany("""
        INSERT INTO quotes (quote, author, tags, page, word_count, scraped_at)
        VALUES (:quote, :author, :tags, :page, :word_count, :scraped_at)
    """, data)

    conn.commit()
    count = cursor.execute("SELECT COUNT(*) FROM quotes").fetchone()[0]
    conn.close()
    print(f"✅ Saved {count} quotes to {db_name}")


save_to_sqlite(quotes_data)

In [ ]:
# Query the database with pandas
conn = sqlite3.connect("quotes.db")

df = pd.read_sql_query("SELECT * FROM quotes", conn)
print(f"Total records: {len(df)}")
print(df.head(3))

# Top authors
print("\n🏆 Top authors:")
print(df.groupby("author")["quote"].count().sort_values(ascending=False).head(5))

# Average word count per author
print("\n📊 Average quote word count per author:")
print(df.groupby("author")["word_count"].mean().sort_values(ascending=False).head(5).round(1))

conn.close()

---
## Part 3 — Export to Formatted Excel

> Professional Excel export with column widths, bold headers, color

In [ ]:
# pip install openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

def export_to_excel(data, filename="quotes_report.xlsx"):
    wb = Workbook()
    ws = wb.active
    ws.title = "Quotes"

    # Header style
    header_font = Font(bold=True, color="FFFFFF")
    header_fill = PatternFill("solid", fgColor="1F4E79")

    headers = ["#", "Quote", "Author", "Tags", "Page", "Words", "Scraped At"]
    col_widths = [5, 60, 20, 30, 6, 7, 20]

    # Write headers
    for col, (header, width) in enumerate(zip(headers, col_widths), start=1):
        cell = ws.cell(row=1, column=col, value=header)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center")
        ws.column_dimensions[get_column_letter(col)].width = width

    # Write data rows
    for i, q in enumerate(data, start=1):
        ws.append([
            i,
            q["quote"],
            q["author"],
            q["tags"],
            q["page"],
            q["word_count"],
            q["scraped_at"]
        ])
        # Wrap text in quote column
        ws.cell(row=i+1, column=2).alignment = Alignment(wrap_text=True)
        ws.row_dimensions[i+1].height = 40

    # Freeze top row
    ws.freeze_panes = "A2"

    # Summary sheet
    ws2 = wb.create_sheet("Summary")
    ws2.append(["Metric", "Value"])

    df_temp = pd.DataFrame(data)
    author_counts = df_temp["author"].value_counts()

    ws2.append(["Total Quotes", len(data)])
    ws2.append(["Unique Authors", df_temp["author"].nunique()])
    ws2.append(["Total Pages Scraped", df_temp["page"].max()])
    ws2.append(["Top Author", author_counts.idxmax()])
    ws2.append(["Top Author Quote Count", int(author_counts.max())])
    ws2.append(["Avg Word Count", round(df_temp["word_count"].mean(), 1)])

    wb.save(filename)
    print(f"✅ Saved formatted Excel to {filename}")


export_to_excel(quotes_data)

---
## Part 4 — Schedule the Scraper

> `schedule` library lets you run functions at intervals — like a cron job in Python  
> `APScheduler` is more powerful for production use

```bash
pip install schedule
```

In [ ]:
# Schedule demo (run for 10 seconds, then stop)
try:
    import schedule

    def scheduled_scrape():
        print(f"[{datetime.now()}] 🔄 Running scheduled scrape...")
        # data = scrape_all_quotes()       # ← uncomment in production
        # save_to_sqlite(data)             # ← and this
        print(f"[{datetime.now()}] ✅ Scrape complete")

    # Schedule options:
    schedule.every(1).minutes.do(scheduled_scrape)    # every minute
    # schedule.every().hour.do(scheduled_scrape)      # every hour
    # schedule.every().day.at("09:00").do(scheduled_scrape)  # daily at 9am

    print("Scheduler running — will fire once in ~5 seconds (demo)")
    scheduled_scrape()   # run once immediately as demo

    # In a real script:
    # while True:
    #     schedule.run_pending()
    #     time.sleep(1)

    print("\n✅ Scheduler ready — uncomment the while loop for production")

except ImportError:
    print("⚠️ schedule not installed. Run: pip install schedule")

---
## Part 5 — Streamlit Dashboard

> Save this as `dashboard.py` and run with: `streamlit run dashboard.py`

```bash
pip install streamlit
```

In [ ]:
# Save this code as dashboard.py and run: streamlit run dashboard.py

DASHBOARD_CODE = '''
import streamlit as st
import pandas as pd
import sqlite3

st.set_page_config(
    page_title="Quotes Dashboard",
    page_icon="💬",
    layout="wide"
)

st.title("💬 Scraped Quotes Dashboard")
st.caption("PixelCraft Infotech | Web Scraping Course")

@st.cache_data
def load_data():
    conn = sqlite3.connect("quotes.db")
    df = pd.read_sql_query("SELECT * FROM quotes", conn)
    conn.close()
    return df

df = load_data()

# Metrics row
col1, col2, col3, col4 = st.columns(4)
col1.metric("Total Quotes", len(df))
col2.metric("Unique Authors", df["author"].nunique())
col3.metric("Pages Scraped", df["page"].max())
col4.metric("Avg Words", round(df["word_count"].mean(), 1))

st.divider()

# Sidebar filters
st.sidebar.header("🔍 Filter Quotes")
authors = ["All"] + sorted(df["author"].unique().tolist())
selected_author = st.sidebar.selectbox("Author", authors)
search_term = st.sidebar.text_input("Search in quotes", "")

# Apply filters
filtered = df.copy()
if selected_author != "All":
    filtered = filtered[filtered["author"] == selected_author]
if search_term:
    filtered = filtered[filtered["quote"].str.contains(search_term, case=False)]

st.subheader(f"Showing {len(filtered)} quotes")

# Display quotes as cards
for _, row in filtered.iterrows():
    with st.expander(f"💬 {row["author"]}"):
        st.write(row["quote"])
        if row["tags"]:
            for tag in row["tags"].split(", "):
                st.badge(tag)
        st.caption(f"Page {row["page"]} · {row["word_count"]} words · Scraped: {row["scraped_at"]}")

st.divider()

# Charts
st.subheader("📊 Author Analysis")
author_counts = df["author"].value_counts().head(10)
st.bar_chart(author_counts)

# Download button
csv_data = df.to_csv(index=False).encode("utf-8")
st.download_button(
    label="📥 Download CSV",
    data=csv_data,
    file_name="quotes_export.csv",
    mime="text/csv"
)
'''

# Save dashboard.py
with open("dashboard.py", "w", encoding="utf-8") as f:
    f.write(DASHBOARD_CODE)

print("✅ dashboard.py saved!")
print("\nTo run the dashboard:")
print("  streamlit run dashboard.py")

---
## 🏆 Final Capstone Challenge

**Build a complete quote research tool:**

1. ✅ Scrape all 100 quotes with stealth mode
2. ✅ Save to SQLite with author, tags, word count, timestamp
3. ✅ Export formatted Excel with Summary sheet
4. ✅ Schedule it to run daily at 9am
5. ✅ Build Streamlit dashboard with filters and download

**Extension challenges:**
- Also scrape each author's biography page and store it
- Add a 'Top Tags' chart to the dashboard
- Add email notification when new quotes are found
- Deploy the dashboard to Streamlit Cloud (free)

### Congratulations — you are now a Web Scraping professional! 🎓

In [ ]:
# Final summary of what you've learned
SKILLS_LEARNED = [
    "Phase 1: requests + BeautifulSoup for static pages",
    "Phase 1: Selenium for JavaScript-rendered pages",
    "Phase 1: Multi-page pagination with headless Chrome",
    "Phase 2: CSV, JSON, pandas — saving and cleaning data",
    "Phase 3: WebDriverWait + expected_conditions — smart waits",
    "Phase 3: TimeoutException, StaleElementReferenceException — error handling",
    "Phase 3: Retry decorators and robust scraping patterns",
    "Phase 4: send_keys, Select, ActionChains — form interaction",
    "Phase 4: Login flows and session management",
    "Phase 4: execute_script — scrolling and JS injection",
    "Phase 4: Infinite scroll handling",
    "Phase 5: UA rotation + random delays — anti-bot bypass",
    "Phase 5: undetected-chromedriver — stealth mode",
    "Phase 5: Scrapy framework — production-grade scraping",
    "Capstone: SQLite storage",
    "Capstone: Formatted Excel export with openpyxl",
    "Capstone: Scheduled scraping with schedule library",
    "Capstone: Streamlit dashboard with charts and filters",
]

print("🎓 Skills mastered in this course:")
print()
for i, skill in enumerate(SKILLS_LEARNED, 1):
    print(f"  {i:2d}. {skill}")

print(f"\n✅ Total: {len(SKILLS_LEARNED)} skills learned!")